In [1]:
mol_type = "6r7m"
n_mol = 2
T = 3.3
x = "Vector{Float64}([])"
comment = ""
bnds = 100.0
rs = 1.4
η = 0.3665
σ_r = 0.3
σ_t = 1.25
overlap_jump = 0.0
overlap_slope = 1.1
delaunay_eps = 100.0

# x = "Vector{Float64}([-162.08661830312784,66.73386522009882,86.90445456748496,37.217231065545484,99.4695870305621,79.08110254187702,-66.02862889330753,58.84672473128092,-4.430459453044266,29.257275608246093,118.7365906843296,75.58061253117084,39.32257178999811,16.883164571009974,61.483907474460906,38.64605615758604,139.52511351241074,68.43924059229049])"
# comment = "selection_1_mid_extension"
# bnds = 250.0

# mol_type = "6r7m"
# n_mol = 3
# T = 3.1
# bnds = 150.0
# x = "Vector{Float64}([4.572368150084601,-3.1187846381217073,3.4498327478360062,133.09753010810374,88.53715000672034,77.13022728781246,-0.7198046903986808,-1.7873703866329462,10.789398888479425,134.15803928988996,82.96834098257952,96.1709753485981,2.3248192124276152,0.7692522899975036,5.536597077810221,128.41567353752285,108.25984562985269,81.82235592650689])"
# comment = "4_81_extension"
# rs = 1.4
# η = 0.3665
# σ_r = 0.3
# σ_t = 1.25
# overlap_jump = 0.0
# overlap_slope = 1.1
# delaunay_eps = 100.0


comment = replace!(comment, " ", "_")
simulation_time_minutes = 12.0 * 60.0
input_specifier = "rwm_ma_$(n_mol)_$(mol_type)"

LoadError: MethodError: no method matching replace!(::String, ::String, ::String)

[0mClosest candidates are:
[0m  replace!(::Any, [91m::Pair...[39m; count)
[0m[90m   @[39m [90mBase[39m [90m[4mset.jl:606[24m[39m
[0m  replace!([91m::Union{Function, Type}[39m, ::Any; count)
[0m[90m   @[39m [90mBase[39m [90m[4mset.jl:648[24m[39m


In [10]:
simulations_per_combination = 50

open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    output_directory = "../Simulations/unsorted_output/$(input_specifier)/"
    for _ in 0:simulations_per_combination-1
        name = "$(i)_$(input_specifier)"
        input_string = escape_string("name=\"$name\";x=$(x);T=$(T);rs=$rs;η=$η;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$bnds;n_mol=$n_mol;mol_type=\"$mol_type\";simulation_time_minutes=$simulation_time_minutes;output_directory=\"$output_directory\";delaunay_eps=$delaunay_eps;comment=\"$comment\"")
        println(io, "$i $input_string")
        i += 1
    end
end

In [11]:
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

50

In [12]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("../$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/rwm_ma_call.jl\"); rwm_ma_call(\"\$config_string\")"))\"")
    # println(io, "")
    # println(io, "# copy back results")
    # println(io, "mkdir -p ~/output/ && cp -r simulation_output/* ~/output/")
end